# Do Warmer Teams Win More at Warmer World Cups?

Hypothesis: teams from warmer climates have an advantage in warmer tournament conditions.

Approach: for each match, identify the winning team's climate classification, then compare warm-climate team win rates in warm vs cool World Cups.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.bbox'] = 'tight'


In [ ]:
df = pd.read_csv('../data/fifa_world_cup_matches_enriched.csv', parse_dates=['Date'], keep_default_na=False)
print(f'{len(df)} matches, {df["Year"].nunique()} tournaments')

# Ensure warm/cool columns are int
for col in ['Warm_Cup', 'Home_Warm_Climate', 'Away_Warm_Climate']:
    df[col] = df[col].astype(int)


## 1. Determine winner climate for each match


In [ ]:
# Identify winner based on scores
df['Home_Won'] = df['Home_Score'] > df['Away_Score']
df['Away_Won'] = df['Away_Score'] > df['Home_Score']
df['Is_Draw'] = df['Home_Score'] == df['Away_Score']

# Winner's climate classification
df['Winner_Climate'] = None
df.loc[df['Home_Won'], 'Winner_Climate'] = df.loc[df['Home_Won'], 'Home_Warm_Climate']
df.loc[df['Away_Won'], 'Winner_Climate'] = df.loc[df['Away_Won'], 'Away_Warm_Climate']
df['Winner_Climate'] = df['Winner_Climate'].astype('Int64')  # nullable int

df['Winner_Climate_Label'] = df['Winner_Climate'].map({1: 'Warm Team Won', 0: 'Cool Team Won'})

print('Non-draw matches with winner climate:', df['Winner_Climate'].notna().sum())
print('Draws:', df['Is_Draw'].sum())

df[['Year','Home_Team','Home_Score','Away_Score','Away_Team','Winner_Climate_Label']].head(10)


## 2. Win rate: warm-climate teams in warm vs cool tournaments


In [ ]:
# Exclude draws - look only at matches with a winner
decided = df[df['Is_Draw'] == False].copy()

# Warm team win rate by tournament type
warm_win_rate = decided.groupby('Warm_Cup').agg(
    Total_Matches=('Winner_Climate', 'count'),
    Warm_Team_Wins=('Winner_Climate', 'sum'),
).reset_index()

warm_win_rate['Warm_Win_Rate'] = (warm_win_rate['Warm_Team_Wins'] / warm_win_rate['Total_Matches'] * 100).round(1)
warm_win_rate['Cool_Team_Wins'] = warm_win_rate['Total_Matches'] - warm_win_rate['Warm_Team_Wins']
warm_win_rate['Tournament'] = warm_win_rate['Warm_Cup'].map({1: 'Warm Cup', 0: 'Cool Cup'})

warm_win_rate[['Tournament','Total_Matches','Warm_Team_Wins','Cool_Team_Wins','Warm_Win_Rate']]


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

plot_data = warm_win_rate.set_index('Tournament')[['Warm_Team_Wins', 'Cool_Team_Wins']]
plot_data.plot(kind='bar', stacked=True, ax=ax, color=['#DA7B42', '#2A7F8C'], edgecolor='white')

for i, (_, row) in enumerate(warm_win_rate.iterrows()):
    ax.text(i, row['Total_Matches'] / 2, f"Warm teams\nwon {row['Warm_Win_Rate']}%",
            ha='center', va='center', fontsize=12, fontweight='bold', color='white')

ax.set_title('Who Wins? Warm vs Cool Climate Teams by Tournament Type')
ax.set_xlabel('')
ax.set_ylabel('Number of Matches')
ax.legend(['Warm Team Won', 'Cool Team Won'])
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()


## 3. Historical trend: warm team win rate over time


In [ ]:
# Per-tournament warm team win rate
yearly = decided.groupby('Year').agg(
    Matches=('Winner_Climate', 'count'),
    Warm_Wins=('Winner_Climate', 'sum'),
    Avg_Temp=('Tournament_Avg_Temp_C', 'first'),
).reset_index()

yearly['Warm_Win_Rate'] = (yearly['Warm_Wins'] / yearly['Matches'] * 100).round(1)
yearly['Is_Warm_Cup'] = yearly['Avg_Temp'] > df['Tournament_Avg_Temp_C'].median()

fig, ax = plt.subplots(figsize=(14, 5))

colors = yearly['Is_Warm_Cup'].map({True: '#DA7B42', False: '#2A7F8C'})
ax.bar(yearly['Year'].astype(str), yearly['Warm_Win_Rate'], color=colors, edgecolor='white')

ax.axhline(y=yearly['Warm_Win_Rate'].mean(), color='black', linestyle='--',
           label=f"Overall average ({yearly['Warm_Win_Rate'].mean():.1f}%)")
ax.set_title('Warm-Climate Team Win Rate by Tournament Year')
ax.set_ylabel('Warm Team Win Rate (%)')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=90)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#DA7B42', label='Warm Cup'),
                   Patch(facecolor='#2A7F8C', label='Cool Cup')]
ax.legend(handles=legend_elements + [ax.get_legend_handles_labels()[0][0]])
plt.tight_layout()


## 4. Does tournament temperature correlate with warm team win rate?


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(yearly['Avg_Temp'], yearly['Warm_Win_Rate'],
           c=yearly['Is_Warm_Cup'].map({True: '#DA7B42', False: '#2A7F8C'}),
           s=yearly['Matches'] * 2, alpha=0.7, edgecolors='black', linewidth=0.5)

# Trend line
z = np.polyfit(yearly['Avg_Temp'], yearly['Warm_Win_Rate'], 1)
p = np.poly1d(z)
x_line = np.linspace(yearly['Avg_Temp'].min(), yearly['Avg_Temp'].max(), 50)
ax.plot(x_line, p(x_line), 'gray', linestyle='--', alpha=0.7,
        label=f"Trend (slope = {z[0]:.2f})")

for _, row in yearly.iterrows():
    ax.annotate(str(int(row['Year'])), (row['Avg_Temp'], row['Warm_Win_Rate']),
                textcoords='offset points', xytext=(5, 5), fontsize=7)

ax.set_title('Warm Team Win Rate vs Tournament Temperature')
ax.set_xlabel('Tournament Average Temperature (C)')
ax.set_ylabel('Warm Team Win Rate (%)')
ax.axhline(y=50, color='gray', linestyle=':', alpha=0.3)
ax.legend()
plt.tight_layout()


## 5. Draw rate: do warm tournaments have different draw rates?


In [ ]:
draw_rate = df.groupby('Warm_Cup').agg(
    Matches=('Is_Draw', 'count'),
    Draws=('Is_Draw', 'sum'),
).reset_index()

draw_rate['Draw_Rate'] = (draw_rate['Draws'] / draw_rate['Matches'] * 100).round(1)
draw_rate['Tournament'] = draw_rate['Warm_Cup'].map({1: 'Warm Cup', 0: 'Cool Cup'})

print('Draw rates by tournament type:')
for _, r in draw_rate.iterrows():
    print(f"  {r['Tournament']}: {r['Draw_Rate']}% ({int(r['Draws'])}/{int(r['Matches'])} matches)")

fig, ax = plt.subplots(figsize=(5, 4))
draw_rate.set_index('Tournament')['Draw_Rate'].plot(
    kind='bar', ax=ax, color=['#2A7F8C', '#DA7B42'], edgecolor='white', rot=0)
ax.set_title('Draw Rate by Tournament Type')
ax.set_xlabel('')
ax.set_ylabel('Draw Rate (%)')
ax.bar_label(ax.containers[0], fmt='%.1f%%')
plt.tight_layout()


## 6. Goals: do warm teams score more in warm cups?


In [ ]:
# Total goals by team climate and tournament type
home_goals = df.groupby(['Warm_Cup', 'Home_Warm_Climate'])['Home_Score'].sum().reset_index()
away_goals = df.groupby(['Warm_Cup', 'Away_Warm_Climate'])['Away_Score'].sum().reset_index()

# Combine: warm team goals = sum of goals by warm home teams + warm away teams
goals_warm = pd.DataFrame({
    'Warm_Cup': [0, 0, 1, 1],
    'Team_Climate': ['Warm', 'Cool', 'Warm', 'Cool'],
    'Goals': [
        home_goals[(home_goals['Warm_Cup'] == 0) & (home_goals['Home_Warm_Climate'] == 1)]['Home_Score'].sum() +
        away_goals[(away_goals['Warm_Cup'] == 0) & (away_goals['Away_Warm_Climate'] == 1)]['Away_Score'].sum(),
        home_goals[(home_goals['Warm_Cup'] == 0) & (home_goals['Home_Warm_Climate'] == 0)]['Home_Score'].sum() +
        away_goals[(away_goals['Warm_Cup'] == 0) & (away_goals['Away_Warm_Climate'] == 0)]['Away_Score'].sum(),
        home_goals[(home_goals['Warm_Cup'] == 1) & (home_goals['Home_Warm_Climate'] == 1)]['Home_Score'].sum() +
        away_goals[(away_goals['Warm_Cup'] == 1) & (away_goals['Away_Warm_Climate'] == 1)]['Away_Score'].sum(),
        home_goals[(home_goals['Warm_Cup'] == 1) & (home_goals['Home_Warm_Climate'] == 0)]['Home_Score'].sum() +
        away_goals[(away_goals['Warm_Cup'] == 1) & (away_goals['Away_Warm_Climate'] == 0)]['Away_Score'].sum(),
    ]
})

# Matches per combination for per-match average
matches_per = df.groupby('Warm_Cup').size()
goals_warm['Goals_Per_Match'] = goals_warm['Goals'] / goals_warm['Warm_Cup'].map({0: matches_per[0], 1: matches_per[1]})
goals_warm['Tournament'] = goals_warm['Warm_Cup'].map({1: 'Warm Cup', 0: 'Cool Cup'})

fig, ax = plt.subplots(figsize=(7, 5))
pivot = goals_warm.pivot_table(values='Goals_Per_Match', index='Tournament', columns='Team_Climate')
pivot.plot(kind='bar', ax=ax, color=['#2A7F8C', '#DA7B42'], edgecolor='white', rot=0)
ax.bar_label(ax.containers[0], fmt='%.2f', fontsize=10)
ax.bar_label(ax.containers[1], fmt='%.2f', fontsize=10)
ax.set_title('Goals per Match by Team Climate and Tournament Type')
ax.set_xlabel('')
ax.set_ylabel('Goals per Match')
ax.legend(title='Team Climate')
plt.tight_layout()


## 7. Summary


In [ ]:
print('=== KEY FINDINGS ===')
print()

warm_rate_cool = warm_win_rate[warm_win_rate['Warm_Cup'] == 0]['Warm_Win_Rate'].values[0]
warm_rate_warm = warm_win_rate[warm_win_rate['Warm_Cup'] == 1]['Warm_Win_Rate'].values[0]
delta = warm_rate_warm - warm_rate_cool

print(f'Warm-climate team win rate:')
print(f'  Cool tournaments: {warm_rate_cool}%')
print(f'  Warm tournaments: {warm_rate_warm}%')
print(f'  Difference: {delta:+.1f} pp')
print()

if delta > 0:
    print(f'CONCLUSION: Warm-climate teams DO win more in warmer World Cups (+{delta:.1f} pp).')
else:
    print(f'CONCLUSION: Warm-climate teams do NOT win more in warmer World Cups ({delta:.1f} pp).')


## 8. Deep Dive: Per-Tournament Breakdown


In [ ]:
# For each tournament, count warm vs cool teams participating
teams_per_cup = df.groupby('Year').agg(
    Warm_Teams=('Home_Warm_Climate', lambda x: set(df.loc[x.index, 'Home_Warm_Climate']) | set(df.loc[x.index, 'Away_Warm_Climate'])),
    Matches=('Home_Score', 'count'),
).reset_index()

teams_per_cup['Total_Teams'] = teams_per_cup['Warm_Teams'].apply(lambda s: len(set(x for xs in s for x in xs)))
teams_per_cup['Warm_Team_Count'] = teams_per_cup['Warm_Teams'].apply(lambda s: sum(1 for x in set(x for xs in s for x in xs) if x == 1))
teams_per_cup['Cool_Team_Count'] = teams_per_cup['Total_Teams'] - teams_per_cup['Warm_Team_Count']
teams_per_cup['Warm_Team_Pct'] = (teams_per_cup['Warm_Team_Count'] / teams_per_cup['Total_Teams'] * 100).round(1)

print('Warm team representation per tournament:')
teams_per_cup[['Year','Total_Teams','Warm_Team_Count','Cool_Team_Count','Warm_Team_Pct']].head(22)


## 9. Expected vs Actual Wins

If warm teams make up X% of participants, we'd expect them to win X% of matches. The difference tells us if they over/underperform.


In [ ]:
# Calculate actual win rates per tournament
actual = decided.groupby('Year').agg(
    Total_Wins=('Winner_Climate', 'count'),
    Warm_Wins=('Winner_Climate', 'sum'),
).reset_index()
actual['Warm_Win_Rate'] = (actual['Warm_Wins'] / actual['Total_Wins'] * 100).round(1)

# Merge with team representation
perf = teams_per_cup[['Year','Total_Teams','Warm_Team_Count','Warm_Team_Pct']].merge(actual, on='Year')
perf['Expected_Warm_Wins'] = (perf['Total_Wins'] * perf['Warm_Team_Pct'] / 100).round(1)
perf['Overperformance'] = (perf['Warm_Wins'] - perf['Expected_Warm_Wins']).round(1)
perf['Tournament_Temp'] = df.groupby('Year')['Tournament_Avg_Temp_C'].first().values.round(1)
perf['Is_Warm_Cup'] = perf['Tournament_Temp'] > 19.3

print('Overperformance: positive = warm teams won more than expected')
perf[['Year','Tournament_Temp','Warm_Team_Pct','Warm_Win_Rate','Expected_Warm_Wins','Warm_Wins','Overperformance']]


In [ ]:
# Visualize overperformance
fig, ax = plt.subplots(figsize=(14, 5))

colors = perf['Overperformance'].apply(lambda x: '#DA7B42' if x > 0 else '#2A7F8C')
ax.bar(perf['Year'].astype(str), perf['Overperformance'], color=colors, edgecolor='white')
ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_title('Warm Team Overperformance: Actual Wins Minus Expected Wins')
ax.set_ylabel('Overperformance (wins)')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=90)
plt.tight_layout()


In [ ]:
# Average overperformance in warm vs cool cups
avg_over = perf.groupby('Is_Warm_Cup')['Overperformance'].mean()
print(f'Average overperformance:')
print(f'  Cool cups: {avg_over[0]:+.1f} wins')
print(f'  Warm cups: {avg_over[1]:+.1f} wins')
print(f'  Difference: {avg_over[1] - avg_over[0]:+.1f} wins')


## 10. Warm Cup Deep Dive: Individual Tournament Analysis


In [ ]:
# Detailed view of each warm World Cup
warm_cups = perf[perf['Is_Warm_Cup']].sort_values('Year')

print('WARM WORLD CUPS - Warm Team Performance')
print('=' * 75)
print(f'{"Year":<6} {"Temp":<7} {"Warm Tms":<10} {"Exp Wins":>10} {"Act Wins":>10} {"Overperf":>10} {"Win Rt":>8}')
print('-' * 65)
for _, r in warm_cups.iterrows():
    print(f'{int(r["Year"]):<6} {r["Tournament_Temp"]:5.1f}C  {r["Warm_Team_Pct"]:5.1f}%    {r["Expected_Warm_Wins"]:>10.1f} {int(r["Warm_Wins"]):>10} {r["Overperformance"]:>+10.1f} {r["Warm_Win_Rate"]:>7.1f}%')

print()
print(f'Average overperformance in warm cups: {warm_cups["Overperformance"].mean():+.1f} wins')
print(f'Warm cups where warm teams overperformed: {(warm_cups["Overperformance"] > 0).sum()}/{len(warm_cups)}')


In [ ]:
# Same for cool cups
cool_cups = perf[~perf['Is_Warm_Cup']].sort_values('Year')

print('COOL WORLD CUPS - Warm Team Performance')
print('=' * 75)
print(f'{"Year":<6} {"Temp":<7} {"Warm Tms":<10} {"Exp Wins":>10} {"Act Wins":>10} {"Overperf":>10} {"Win Rt":>8}')
print('-' * 65)
for _, r in cool_cups.iterrows():
    print(f'{int(r["Year"]):<6} {r["Tournament_Temp"]:5.1f}C  {r["Warm_Team_Pct"]:5.1f}%    {r["Expected_Warm_Wins"]:>10.1f} {int(r["Warm_Wins"]):>10} {r["Overperformance"]:>+10.1f} {r["Warm_Win_Rate"]:>7.1f}%')

print()
print(f'Average overperformance in cool cups: {cool_cups["Overperformance"].mean():+.1f} wins')
print(f'Cool cups where warm teams overperformed: {(cool_cups["Overperformance"] > 0).sum()}/{len(cool_cups)}')


## 11. Stage Progression: Do warm teams go further in warm cups?


In [ ]:
# Count unique warm vs cool teams reaching each stage per tournament type
stages_order = ['Group Stage', 'Round of 16', 'Quarter-final', 'Semi-final', 'Final']

stage_data = []
for warm_cup in [0, 1]:
    cup_df = df[df['Warm_Cup'] == warm_cup]
    for stage in stages_order:
        stage_matches = cup_df[cup_df['Stage'] == stage]
        warm_teams_in_stage = len(set(stage_matches['Home_Team'][stage_matches['Home_Warm_Climate'] == 1]) |
                                  set(stage_matches['Away_Team'][stage_matches['Away_Warm_Climate'] == 1]))
        cool_teams_in_stage = len(set(stage_matches['Home_Team'][stage_matches['Home_Warm_Climate'] == 0]) |
                                  set(stage_matches['Away_Team'][stage_matches['Away_Warm_Climate'] == 0]))
        stage_data.append({
            'Tournament': 'Warm Cup' if warm_cup else 'Cool Cup',
            'Stage': stage,
            'Warm_Teams': warm_teams_in_stage,
            'Cool_Teams': cool_teams_in_stage,
        })

stage_df = pd.DataFrame(stage_data)
stage_df['Warm_Pct'] = (stage_df['Warm_Teams'] / (stage_df['Warm_Teams'] + stage_df['Cool_Teams']) * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, (label, data) in zip(axes, [('Cool Cup', stage_df[stage_df['Tournament'] == 'Cool Cup']),
                                       ('Warm Cup', stage_df[stage_df['Tournament'] == 'Warm Cup'])]):
    ax.barh(data['Stage'], data['Warm_Teams'], color='#DA7B42', label='Warm Teams')
    ax.barh(data['Stage'], data['Cool_Teams'], left=data['Warm_Teams'], color='#2A7F8C', label='Cool Teams')
    for i, (_, row) in enumerate(data.iterrows()):
        ax.text(row['Warm_Teams'] + row['Cool_Teams'] + 2, i,
                f"{row['Warm_Pct']}% warm", va='center', fontsize=9)
    ax.set_title(f'{label} - Teams Reaching Each Stage')
    ax.set_xlabel('Number of Unique Teams')
    if ax == axes[0]:
        ax.legend(loc='lower right')

fig.suptitle('Stage Progression: Warm vs Cool Teams', fontsize=14, y=1.02)
plt.tight_layout()


## 12. Goals For/Against by Team Climate


In [ ]:
# Average goals scored and conceded by warm vs cool teams
goals_summary = []
for warm_cup in [0, 1]:
    cup_df = df[df['Warm_Cup'] == warm_cup]
    cup_label = 'Warm Cup' if warm_cup else 'Cool Cup'
    
    for team_climate, climate_label in [(1, 'Warm'), (0, 'Cool')]:
        # Goals scored by these teams (when home + when away)
        scored_home = cup_df[cup_df['Home_Warm_Climate'] == team_climate]['Home_Score'].sum()
        scored_away = cup_df[cup_df['Away_Warm_Climate'] == team_climate]['Away_Score'].sum()
        
        # Goals conceded by these teams (opponent's goals)
        conceded_home = cup_df[cup_df['Home_Warm_Climate'] == team_climate]['Away_Score'].sum()
        conceded_away = cup_df[cup_df['Away_Warm_Climate'] == team_climate]['Home_Score'].sum()
        
        matches_played = len(cup_df[cup_df['Home_Warm_Climate'] == team_climate]) + len(cup_df[cup_df['Away_Warm_Climate'] == team_climate])
        
        goals_summary.append({
            'Tournament': cup_label,
            'Team_Climate': climate_label,
            'Goals_Scored': scored_home + scored_away,
            'Goals_Conceded': conceded_home + conceded_away,
            'Matches_Played': matches_played,
            'Goals_Per_Match': round((scored_home + scored_away) / matches_played, 2),
            'Conceded_Per_Match': round((conceded_home + conceded_away) / matches_played, 2),
        })

goals_df = pd.DataFrame(goals_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for i, metric in enumerate(['Goals_Per_Match', 'Conceded_Per_Match']):
    pivot = goals_df.pivot_table(values=metric, index='Tournament', columns='Team_Climate')
    pivot.plot(kind='bar', ax=axes[i], color=['#2A7F8C', '#DA7B42'], edgecolor='white', rot=0)
    axes[i].set_title(f'Average {metric.replace("_", " ").title()}')
    axes[i].set_xlabel('')
    axes[i].legend(title='Team Climate')
    for container in axes[i].containers:
        axes[i].bar_label(container, fmt='%.2f', fontsize=10)

fig.suptitle('Goals For and Against by Team Climate and Tournament Type', fontsize=13, y=1.02)
plt.tight_layout()


## 13. Overall Summary


In [ ]:
print('=== COMPREHENSIVE SUMMARY ===')
print()

cool_win = warm_win[warm_win['Warm_Cup'] == 0]['Warm_Win_Rate'].values[0]
warm_win_rate = warm_win[warm_win['Warm_Cup'] == 1]['Warm_Win_Rate'].values[0]

print(f'1. WIN RATE: Warm teams win {cool_win}% in cool cups vs {warm_win_rate}% in warm cups')
print(f'   Delta: {warm_win_rate - cool_win:+.1f} pp')
print()

ov_cool = perf[~perf['Is_Warm_Cup']]['Overperformance'].mean()
ov_warm = perf[perf['Is_Warm_Cup']]['Overperformance'].mean()
print(f'2. OVERPERFORMANCE (actual minus expected wins):')
print(f'   Cool cups: {ov_cool:+.1f}, Warm cups: {ov_warm:+.1f}')
print()

print(f'3. WARM CUPS won by warm teams (win rate > 50%):')
warm_over50 = perf[perf['Is_Warm_Cup'] & (perf['Warm_Win_Rate'] > 50)]
print(f'   {len(warm_over50)}/{len(perf[perf["Is_Warm_Cup"]])} warm cups')
for _, r in warm_over50.iterrows():
    print(f'   {int(r["Year"])} - warm teams won {r["Warm_Win_Rate"]}% of matches')
print()

print(f'4. COOL CUPS won by warm teams:')
cool_over50 = perf[~perf['Is_Warm_Cup'] & (perf['Warm_Win_Rate'] > 50)]
print(f'   {len(cool_over50)}/{len(perf[~perf["Is_Warm_Cup"]])} cool cups')
for _, r in cool_over50.iterrows():
    print(f'   {int(r["Year"])} - warm teams won {r["Warm_Win_Rate"]}% of matches')
print()

print(f'5. DRAW RATE: Cool {draw_rate[0]:.1f}% vs Warm {draw_rate[1]:.1f}%')

print()
if warm_win_rate > cool_win:
    print('CONCLUSION: Warm-climate teams DO win more in warmer World Cups.')
    print(f'The effect is modest ({warm_win_rate - cool_win:+.1f} pp) but consistent.')
else:
    print('CONCLUSION: No evidence that warm-climate teams win more in warmer World Cups.')


---

## ELO-ADJUSTED ANALYSIS

Controlling for team strength using Elo ratings. For each match, we compare the actual outcome against what Elo predicted.


In [ ]:
# The enriched dataset now includes:
#   Home_Elo, Away_Elo, Home_Elo_Advantage, Home_Elo_Expected
# Home_Elo_Expected = probability home team wins (0-1) based on Elo difference

print('Elo columns:', [c for c in df.columns if 'Elo' in c])
df[['Year','Home_Team','Away_Team','Home_Elo','Away_Elo','Home_Elo_Expected']].head(10)


In [ ]:
# Compute Elo-expected warm team success for each match
# If the home team is warm-climate, use Home_Elo_Expected
# If the away team is warm-climate, use (1 - Home_Elo_Expected)

df['Warm_Elo_Expected'] = np.where(
    df['Home_Warm_Climate'] == 1,
    df['Home_Elo_Expected'],
    1 - df['Home_Elo_Expected']
)

# Actual warm team success: 1 = warm win, 0 = cool win, 0.5 = draw
df['Warm_Elo_Actual'] = np.where(
    df['Is_Draw'] == 1, 0.5,
    np.where(df['Winner_Climate'] == 1, 1.0, 0.0)
)

df['Warm_Elo_Overperformance'] = df['Warm_Elo_Actual'] - df['Warm_Elo_Expected']


In [ ]:
# Group by tournament warmth
elo_analysis = df.groupby('Warm_Cup').agg(
    Matches=('Warm_Elo_Overperformance', 'count'),
    Avg_Expected=('Warm_Elo_Expected', 'mean'),
    Avg_Actual=('Warm_Elo_Actual', 'mean'),
    Avg_Overperformance=('Warm_Elo_Overperformance', 'mean'),
    Total_Overperformance=('Warm_Elo_Overperformance', 'sum'),
).reset_index()
elo_analysis['Tournament'] = elo_analysis['Warm_Cup'].map({1: 'Warm Cup', 0: 'Cool Cup'})

print('=== ELO-ADJUSTED WARM TEAM PERFORMANCE ===')
print()
for _, r in elo_analysis.iterrows():
    print(f'{r["Tournament"]}:')
    print(f'  Elo-expected warm team win prob: {r["Avg_Expected"]:.3f}')
    print(f'  Actual warm team win prob:       {r["Avg_Actual"]:.3f}')
    print(f'  Overperformance per match:       {r["Avg_Overperformance"]:+.3f}')
    print(f'  Total overperformance:           {r["Total_Overperformance"]:+.1f} wins')
    print()

cool_op = elo_analysis[elo_analysis['Warm_Cup'] == 0]['Avg_Overperformance'].values[0]
warm_op = elo_analysis[elo_analysis['Warm_Cup'] == 1]['Avg_Overperformance'].values[0]
print(f'Net Elo-adjusted advantage in warm cups: {warm_op - cool_op:+.3f} per match')


In [ ]:
# Visualize Elo-adjusted overperformance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
elo_analysis.set_index('Tournament')['Avg_Overperformance'].plot(
    kind='bar', ax=ax, color=['#2A7F8C', '#DA7B42'], edgecolor='white', rot=0)
ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_title('Elo-Adjusted Warm Team Overperformance per Match')
ax.set_xlabel('')
ax.set_ylabel('Overperformance (actual - expected)')
for c in ax.containers:
    ax.bar_label(c, fmt='%+.3f', fontsize=12)

# Per-tournament Elo overperformance
ax2 = axes[1]
decided = df[df['Is_Draw'] == 0]
yearly_elo = decided.groupby('Year').agg(
    Matches=('Warm_Elo_Expected', 'count'),
    Expected=('Warm_Elo_Expected', 'sum'),
    Actual=('Winner_Climate', lambda x: (x == 1).sum()),
    Avg_Temp=('Tournament_Avg_Temp_C', 'first'),
).reset_index()
yearly_elo.columns = ['Year', 'Matches', 'Expected', 'Actual', 'Avg_Temp']
yearly_elo['Overperformance'] = yearly_elo['Actual'] - yearly_elo['Expected']
yearly_elo['Is_Warm'] = yearly_elo['Avg_Temp'] > 19.3

colors = yearly_elo['Overperformance'].apply(lambda x: '#DA7B42' if x > 0 else '#2A7F8C')
ax2.bar(yearly_elo['Year'].astype(str), yearly_elo['Overperformance'], color=colors, edgecolor='white')
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.set_title('Elo-Adjusted Overperformance by Tournament')
ax2.set_ylabel('Warm Team Wins Above Elo Expectation')
ax2.set_xlabel('')
ax2.tick_params(axis='x', rotation=90)

plt.tight_layout()


In [ ]:
# Warm teams: overperformance vs tournament temperature
fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(yearly_elo['Avg_Temp'], yearly_elo['Overperformance'],
           c=yearly_elo['Is_Warm'].map({True: '#DA7B42', False: '#2A7F8C'}),
           s=yearly_elo['Matches'] * 3, alpha=0.7, edgecolors='black', linewidth=0.5)

# Trend line
z = np.polyfit(yearly_elo['Avg_Temp'], yearly_elo['Overperformance'], 1)
p = np.poly1d(z)
x_line = np.linspace(yearly_elo['Avg_Temp'].min(), yearly_elo['Avg_Temp'].max(), 50)
ax.plot(x_line, p(x_line), 'gray', linestyle='--', alpha=0.7, label=f'Slope = {z[0]:.2f}')

for _, row in yearly_elo.iterrows():
    ax.annotate(str(int(row['Year'])), (row['Avg_Temp'], row['Overperformance']),
                textcoords='offset points', xytext=(5, 5), fontsize=8)

ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_title('Warm Team Elo-Adjusted Overperformance vs Tournament Temperature')
ax.set_xlabel('Tournament Avg Temperature (C)')
ax.set_ylabel('Warm Team Wins Above/Below Elo Expectation')
ax.legend()
plt.tight_layout()


In [ ]:
# Key insight: goals vs Elo expectation
print('=== FINAL SUMMARY (ELO-ADJUSTED) ===')
print()
print('After controlling for team strength (Elo), warm-climate teams:')
print(f'  In cool World Cups: {cool_op:+.3f} per match ({elo_analysis[elo_analysis["Warm_Cup"]==0]["Total_Overperformance"].values[0]:+.1f} wins total)')
print(f'  In warm World Cups: {warm_op:+.3f} per match ({elo_analysis[elo_analysis["Warm_Cup"]==1]["Total_Overperformance"].values[0]:+.1f} wins total)')
print()
print(f'  Net advantage: {(warm_op - cool_op):+.3f} per match')
print()
warm_over = yearly_elo[yearly_elo['Is_Warm']]['Overperformance'].mean()
cool_over = yearly_elo[~yearly_elo['Is_Warm']]['Overperformance'].mean()
print(f'  Per-tournament avg overperformance:')
print(f'    Warm cups: {warm_over:+.1f} wins')
print(f'    Cool cups: {cool_over:+.1f} wins')
print(f'    Difference: {warm_over - cool_over:+.1f} wins')
print()
print(f'  Warm cups where warm teams beat Elo expectation:')
warm_pos = yearly_elo[yearly_elo['Is_Warm'] & (yearly_elo['Overperformance'] > 0)]
print(f'    {len(warm_pos)}/{len(yearly_elo[yearly_elo["Is_Warm"]])} warm cups')
print(f'  Cool cups where warm teams beat Elo expectation:')
cool_pos = yearly_elo[~yearly_elo['Is_Warm'] & (yearly_elo['Overperformance'] > 0)]
print(f'    {len(cool_pos)}/{len(yearly_elo[~yearly_elo["Is_Warm"]])} cool cups')
print()
if warm_op > cool_op:
    print('CONCLUSION: After controlling for Elo, warm-climate teams DO outperform in warmer World Cups.')
    print(f'  The effect is {warm_op - cool_op:+.3f} per match ({(warm_op - cool_op) * 100:.1f}% improvement).')
else:
    print('CONCLUSION: No Elo-adjusted advantage for warm teams in warm cups.')
